In [1]:
# librerias

!pip install pandas openpyxl -q
!pip install folium -q
!pip install geopy -q
!pip install opencage -q


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import pandas as pd
import warnings
import folium
import numpy as np
from tqdm import tqdm
import requests
import time
import re
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from opencage.geocoder import OpenCageGeocode
import pandas as pd
import json
from time import sleep
from pathlib import Path
from openai import OpenAI
from geopy.distance import geodesic


warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [ ]:
df = pd.read_excel("data/Reporte Histórico de Roturas - Surtigas.xlsx", sheet_name="Detalle Roturas Historico")
df.rename(columns={"LAT ": "LAT"}, inplace=True)
df_plano = df.dropna(subset=["LAT", "LONG"])

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=api_key_gpt,
)

In [ ]:
apikey = apikey_geocoder
geocoder = OpenCageGeocode(apikey)

apikey_location_li = apikey_location_li 

def normalizar_direccion(direccion: str, ciudad: str, departamento: str) -> str:
    """Usa Claude para limpiar y estandarizar la dirección colombiana."""
    prompt = f"""
    Normaliza esta dirección colombiana al formato estándar para geocodificación.
    
    Dirección original: {direccion}
    Ciudad: {ciudad}
    Departamento: {departamento}
    
    Reglas:
    - KR o CRA → Carrera
    - CL → Calle
    - TV → Transversal
    - DG → Diagonal
    - El número después del guion son metros desde la intersección
    - MZ/LT son manzana/lote (barrios sin nomenclatura vial)
    - MEMB/MIRO/SC son referencias de barrio, ignóralas si no aportan
    
    Responde SOLO con la dirección normalizada lista para buscar en un mapa.
    Ejemplo: "Carrera 15 # 46-145, Montería, Córdoba, Colombia"
    Si la dirección es ambigua (MZ/LT sin barrio reconocible), responde solo con:
    "ciudad, departamento, Colombia"
    """
    
    response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=150,
    temperature=0,
    )

    return response.choices[0].message.content.strip()


# ── 2. Geocoder con Nominatim ─────────────────────────────────────────────────
geolocator = Nominatim(user_agent="costa_colombia_geocoder_v1")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

def geocodificar_nominatim(query: str, pais: str = "Colombia"):
    """Intenta geocodificar con Nominatim/OSM."""
    try:
        location = geocode(
            f"{query}, {pais}",
            country_codes="co",
            timeout=10
        )
        lat = location.latitude
        lon = location.longitude
        if 5.5 < lat < 10.8 and -77.5 < lon < -73:  # bounding box Colombia
            return lat, lon, "nominatim"
    except Exception:
        pass
    return None, None, None


# ── 3. Fallback: geocode.xyz ──────────────────────────────────────────────────
def geocodificar_geocodexyz(query: str):
    """Geocodificador alternativo con soporte Colombia."""
    try:
        url = "https://geocode.xyz"
        params = {"locate": query, "json": 1, "country": "CO"}
        r = requests.get(url, params=params, timeout=10)
        data = r.json()
        lat = float(data.get("latt", 0))
        lon = float(data.get("longt", 0))
        if 5.5 < lat < 10.8 and -77.5 < lon < -73:  # bounding box Colombia
            return lat, lon, "geocode.xyz"
    except Exception:
        pass
    return None, None, None

# ── 3.5 Fallback: opencage opcion de pago ──────────────────────────────────────────────────
def geocodificar_opencage(query: str):
    """Geocodificador alternativo de pago con soporte Colombia."""
    try:
        results = geocoder.geocode(query)
        lat = float(results[0]['geometry']['lat'])
        lon = float(results[0]['geometry']['lng'])
        if 5.5 < lat < 10.8 and -77.5 < lon < -73:  # bounding box Colombia
            return lat, lon, "opencode"
    except Exception:
        pass
    return None, None, None

def geocode_locationiq(query):
    url = "https://us1.locationiq.com/v1/search"
    params = {
        "key": apikey_location,
        "q": query,
        "format": "json"
    }
    try:
        r = requests.get(url, params=params)
        data = r.json()
        lat = float(data[0]["lat"])
        lon = float(data[0]["lon"])
        if 5.5 < lat < 10.8 and -77.5 < lon < -73:  # bounding box Colombia
            return lat, lon, "locationiq"
    except Exception:
        pass
    return None, None, None


In [4]:
def geocodificar(row: dict) -> tuple:
    query_normalizada = row.get("direccion_normalizada", "")
    ciudad = row.get("Ciudad", "")

    # Paso 3.5: opencode
    lat, lon, fuente = geocodificar_opencage(query_normalizada)
    if lat is not None:
       return lat, lon, fuente, query_normalizada
    
    # Paso 3.8: locationiq
    lat, lon, fuente = geocode_locationiq(query_normalizada)
    if lat is not None:
        return lat, lon, fuente, query_normalizada

    # Paso 3: geocode.xyz
    lat, lon, fuente = geocodificar_geocodexyz(query_normalizada)
    if lat is not None:
        return lat, lon, fuente, query_normalizada

    # Paso 2: Nominatim
    lat, lon, fuente = geocodificar_nominatim(query_normalizada)
    if lat is not None:
        return lat, lon, fuente, query_normalizada
        
    # Paso 4: Centroide del municipio
    if lat is None:
        return np.nan, np.nan, "sin_geocodificar", query_normalizada
    
    return lat, lon, fuente, query_normalizada

In [ ]:
def eval_haversine(A,B):
    distancia_m = geodesic(A,B).meters
    return distancia_m

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
BATCH_SIZE  = 50
SLEEP_TIME  = 2
CHECKPOINT_PATH = Path("checkpoint_normalizacion.json")

# Orden de prioridad: si el primero falla por tokens/rate, intenta el siguiente
LLM_FALLBACK = [
    {"client_type": "openai", "model": "gpt-4.1"},
    {"client_type": "openai", "model": "gpt-4.1-mini"},
    {"client_type": "openai", "model": "gpt-4o-mini"},
    {"client_type": "deepseek", "model": "gpt-4o"}, 
    {"client_type": "openai",   "model": "DeepSeek-V3-0324"},
    {"client_type": "openai",   "model": "grok-3"},
    {"client_type": "openai",   "model": "gpt-4.1-nano"},
]

# ─────────────────────────────────────────────────────────────
# CHECKPOINT: guardar y cargar progreso
# ─────────────────────────────────────────────────────────────
def guardar_checkpoint(resultados: dict):
    """Serializa el dict {idx: direccion} a disco."""
    CHECKPOINT_PATH.write_text(
        json.dumps({str(k): v for k, v in resultados.items()}, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    print(f"  💾 Checkpoint guardado ({len(resultados)} registros)")

def cargar_checkpoint() -> dict:
    """Carga checkpoint si existe. Retorna dict vacío si no hay."""
    if CHECKPOINT_PATH.exists():
        data = json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
        resultado = {int(k): v for k, v in data.items()}  # restaurar claves como int
        print(f"♻️  Checkpoint encontrado: {len(resultado)} registros ya procesados → se saltarán")
        return resultado
    return {}

def borrar_checkpoint():
    """Llamar al final cuando todo salió bien."""
    if CHECKPOINT_PATH.exists():
        CHECKPOINT_PATH.unlink()
        print("🗑️  Checkpoint eliminado (proceso completado)")

# ─────────────────────────────────────────────────────────────
# LIMPIEZA DE JSON
# ─────────────────────────────────────────────────────────────
def limpiar_json(contenido: str) -> list:
    contenido = contenido.strip()
    contenido = re.sub(r"^```(?:json)?\s*", "", contenido)
    contenido = re.sub(r"\s*```$",           "", contenido)
    return json.loads(contenido.strip())

# ─────────────────────────────────────────────────────────────
# LLAMADA AL LLM CON FALLBACK AUTOMÁTICO
# ─────────────────────────────────────────────────────────────
modelos_bloqueados = set()

def llamar_llm_con_fallback(prompt: str) -> tuple[str, str]:
    """
    Intenta cada modelo en LLM_FALLBACK hasta que uno responda.
    Retorna (contenido_respuesta, modelo_usado).
    Lanza RuntimeError si todos fallan.
    """
    errores = []

    for config in LLM_FALLBACK:
        model = config["model"]

        if model in modelos_bloqueados:
            print(f"⏭️ {model} bloqueado, saltando...")
            continue

        try:
            print(f"  🤖 Intentando con {model}...", end=" ")
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                timeout=30,
            )
            contenido = response.choices[0].message.content
            print(f"✅")
            return contenido, model

        except Exception as e:
            msg = str(e)
            # Errores que justifican saltar al siguiente modelo
            fatales = [
                "insufficient_quota", "rate_limit", "context_length",
                "max_tokens", "token", "overloaded", "capacity",
                "model_not_found", "does not exist", "Request timed out",
                "RateLimitReached", "429", "RateLimitError", "UserByModelByDay"
            ]
            es_fatal = any(palabra in msg.lower() for palabra in fatales)
            errores.append(f"{model}: {msg[:80]}")
            print(f"❌ ({msg[:60]})")
            if "RateLimitError" in msg:
                print(f"⏭️ Saltando modelo {model} por límite")
                continue
            elif es_fatal:
                modelos_bloqueados.add(model)
                continue          # pasar al siguiente modelo
            else:
                raise             # error inesperado, propagar

    raise RuntimeError(
        f"❌ Todos los modelos fallaron:\n" + "\n".join(f"  - {e}" for e in errores)
    )

# ─────────────────────────────────────────────────────────────
# BATCH: normalizar un lote
# ─────────────────────────────────────────────────────────────
def normalizar_batch(df_batch: pd.DataFrame) -> tuple[dict, str]:
    """Retorna (dict {idx_real: direccion_normalizada}, modelo_usado)."""
    
    registros = []
    id_map    = {}

    for local_id, (idx_real, row) in enumerate(df_batch.iterrows()):
        id_map[local_id] = idx_real
        registros.append({
            "id":          local_id,
            "direccion":   str(row.get("Dirección", "")).strip(),
            "ciudad":      str(row.get("Ciudad",    "")).strip(),
        })

    prompt = f"""
Normaliza las siguientes direcciones colombianas para geocodificación.

Reglas de normalización:
- KR, CRA → Carrera | CL → Calle | TV → Transversal | DG → Diagonal
- Formato esperado: "Carrera X # Y-Z, Ciudad"
- MZ/LT sin barrio claro → usar solo "Ciudad"
- Ignorar prefijos irrelevantes: MEMB, MIRO, SC, etc.
- NORTE/SUR/ESTE/OESTE → conservarlos (ej: "Carrera 2 Norte # 3C Norte-55")
- Carrera o(or) Calle: solo uno debe prevalecer en la dirección (ej: "Carrera 10 #18-37", "Calle 67 #67-110)
- Devuelve EXACTAMENTE los mismos IDs recibidos, sin omitir ninguno.

Responde ÚNICAMENTE con un array JSON válido, sin explicaciones ni markdown:
[
  {{"id": 0, "direccion_normalizada": "Carrera 2 #3C-55, Ciudad"}},
  {{"id": 1, "direccion_normalizada": "..."}}
]

Direcciones a normalizar:
{json.dumps(registros, ensure_ascii=False, indent=2)}
"""

    contenido, modelo = llamar_llm_con_fallback(prompt)

    try:
        salida = limpiar_json(contenido)
        return (
            {id_map[item["id"]]: item["direccion_normalizada"]
             for item in salida if item["id"] in id_map},
            modelo
        )
    except json.JSONDecodeError:
        print(f"  ⚠️  JSON inválido:\n{contenido[:300]}")
        return {}, modelo

# ─────────────────────────────────────────────────────────────
# PROCESAMIENTO POR LOTES CON CHECKPOINT
# ─────────────────────────────────────────────────────────────
def procesar_en_batches(df: pd.DataFrame) -> pd.DataFrame:
    total      = len(df)
    resultados = cargar_checkpoint()          # ← retoma donde quedó
    modelos_usados = {}                       # tracking de qué modelo procesó qué batch

    # Calcular qué índices ya están resueltos
    indices_pendientes = [i for i in range(0, total, BATCH_SIZE)
                          if not all(idx in resultados
                                     for idx in df.iloc[i:i+BATCH_SIZE].index)]

    print(f"📋 Total filas     : {total}")
    print(f"✅ Ya procesadas   : {len(resultados)}")
    print(f"⏳ Batches restantes: {len(indices_pendientes)}\n")

    for i in indices_pendientes:
        df_batch    = df.iloc[i : i + BATCH_SIZE]
        num_batch   = i // BATCH_SIZE + 1
        print(f"📦 Batch {num_batch} | filas {i}–{i+len(df_batch)-1} de {total}")

        try:
            parcial, modelo = normalizar_batch(df_batch)
            modelos_usados[num_batch] = modelo

            resultados.update(parcial)
            guardar_checkpoint(resultados)     # ← guardar tras cada batch exitoso

            perdidos = set(df_batch.index) - set(parcial.keys())
            if perdidos:
                print(f"  ⚠️  {len(perdidos)} registros sin normalizar en este batch")

        except RuntimeError as e:
            # Todos los modelos fallaron → guardar lo que hay y abortar
            print(f"\n🚨 {e}")
            print(f"💾 Progreso guardado en {CHECKPOINT_PATH}. Vuelve a ejecutar cuando tengas créditos.")
            break

        sleep(SLEEP_TIME)

    # Mapear al DataFrame
    df["direccion_normalizada"] = df.index.map(resultados)

    mascara_nulos = df["direccion_normalizada"].isna()
    df.loc[mascara_nulos, "direccion_normalizada"] = (
        df.loc[mascara_nulos, "Ciudad"] + ", " +
        df.loc[mascara_nulos, "Distrito"] + ", Colombia"
    )

    print(f"\n📊 Resumen final:")
    print(f"   ✅ Normalizados con LLM : {(~mascara_nulos).sum()}")
    print(f"   ⚠️  Fallback ciudad      : {mascara_nulos.sum()}")
    if modelos_usados:
        from collections import Counter
        conteo = Counter(modelos_usados.values())
        print(f"   🤖 Modelos usados       : {dict(conteo)}")

    # Solo borrar checkpoint si se procesó todo
    if mascara_nulos.sum() == 0 or not indices_pendientes:
        borrar_checkpoint()

    return df

In [ ]:
# ─────────────────────────────────────────────────────────────
# EJECUCIÓN OJOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO
# ─────────────────────────────────────────────────────────────

df_nan = df[df[["LAT", "LONG"]].isna().any(axis=1)]
df_nan = df_nan.sample(frac=1, random_state=42).reset_index(drop=True)


df_nan = procesar_en_batches(df_nan)
#df_nan.to_csv("df_dir_good.csv")

In [ ]:
# ─────────────────────────────────────────────────────────────
# LECTURA DE DIRECCION PARCIADA POR IA
# ─────────────────────────────────────────────────────────────

df_nan = pd.read_csv("df_dir_good.csv")
df_nan.rename(columns={"LAT ": "LAT"}, inplace=True)

df_nan = df_nan.dropna(subset=["direccion_normalizada"])
df_nan.shape

In [ ]:
# ──────────────────────────────────────────────────────────────────
# GEOCODIFICACIÓN
# ──────────────────────────────────────────────────────────────────

# ══════════════════════════════════════════════════════════════════
# CONFIGURACIÓN
# ══════════════════════════════════════════════════════════════════
CHECKPOINT_PATH = "checkpoint_geocoding.json"
BATCH_SAVE      = 500

# ══════════════════════════════════════════════════════════════════
# FUNCIONES
# ══════════════════════════════════════════════════════════════════

def cargar_checkpoint(path: str) -> dict:
    """
    Carga resultados previos desde un archivo JSON de checkpoint.
    Retorna un dict vacío si el archivo no existe.
    """
    if not os.path.exists(path):
        print("🆕 Sin checkpoint previo, iniciando desde cero")
        return {}

    with open(path, "r", encoding="utf-8") as f:
        resultados = json.load(f)

    print(f"🔄 Cargados {len(resultados)} registros previos")
    return resultados


def guardar_checkpoint(resultados: dict, path: str) -> None:
    """
    Persiste el diccionario de resultados en el archivo JSON de checkpoint.
    """
    with open(path, "w", encoding="utf-8") as f:
        json.dump(resultados, f, ensure_ascii=False, indent=2)

    print(f"💾 Guardados {len(resultados)} registros en '{path}'")


def filtrar_pendientes(df, resultados: dict):
    """
    Retorna solo las filas del DataFrame cuyo índice
    no está aún en el checkpoint.
    """
    pendientes = df[~df.index.astype(str).isin(resultados.keys())]
    print(f"🧠 Pendientes por procesar: {len(pendientes)}")
    return pendientes


def procesar_fila(row) -> dict:
    """
    Geocodifica una fila y retorna el resultado como dict.
    Adapta esto si tu función geocodificar() cambia de firma.
    """
    lat, lon, fuente, query = geocodificar(row)
    return {"lat": lat, "lon": lon, "fuente": fuente, "query": query}


def ejecutar_geocodificacion(
    df,
    checkpoint_path: str = CHECKPOINT_PATH,
    batch_save: int       = BATCH_SAVE,
) -> dict:
    """
    Orquesta el proceso completo de geocodificación con checkpoint.

    Parámetros
    ----------
    df             : DataFrame con los registros a geocodificar.
    checkpoint_path: Ruta del archivo JSON de checkpoint.
    batch_save     : Cada cuántos registros se guarda el checkpoint.

    Retorna
    -------
    dict con todos los resultados (previos + nuevos).
    """
    resultados = cargar_checkpoint(checkpoint_path)
    pendientes = filtrar_pendientes(df, resultados)

    if pendientes.empty:
        print("✅ Todos los registros ya están geocodificados")
        return resultados

    for contador, (idx, row) in enumerate(
        tqdm(pendientes.iterrows(), total=len(pendientes)), start=1
    ):
        resultados[str(idx)] = procesar_fila(row)

        if contador % batch_save == 0:
            guardar_checkpoint(resultados, checkpoint_path)

    # Guardado final (cubre el remanente que no llegó al batch)
    guardar_checkpoint(resultados, checkpoint_path)
    print("✅ Proceso completado")

    return resultados


# ══════════════════════════════════════════════════════════════════
# EJECUCIÓN  ← única línea que necesitas correr
# ══════════════════════════════════════════════════════════════════
resultados = ejecutar_geocodificacion(df_nan)


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
🔄 Cargados 3500 registros previos
🧠 Pendientes por procesar: 5100


 10%|▉         | 500/5100 [06:09<55:57,  1.37it/s]  


💾 Guardados 4000 registros


 20%|█▉        | 1000/5100 [12:20<39:43,  1.72it/s] 


💾 Guardados 4500 registros


...


💾 Guardados 8000 registros


 98%|█████████▊| 5000/5100 [1:05:30<00:59,  1.67it/s]


💾 Guardados 8500 registros


100%|██████████| 5100/5100 [1:06:57<00:00,  1.27it/s]

✅ Proceso completado


In [ ]:
# ──────────────────────────────────────────────────────────────────
# UNION DE LOS DATAFRAME
# ──────────────────────────────────────────────────────────────────

# Cargar archivo
with open("checkpoint_geocoding.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Convertir a DataFrame
df_resultados = pd.DataFrame.from_dict(data, orient="index")

# Convertir índice a entero (opcional pero recomendado)
df_resultados.index = df_resultados.index.astype(int)

resultados.to_csv("df_dir_inferidos.csv")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# UNION DE LOS DATAFRAME #2
# ──────────────────────────────────────────────────────────────────

df_final = df_nan.join(df_resultados)
df_final["Orgen_data"] = str("Inferido")
df_final[['Ciudad','Diametro','Dimensión Rotura','TERCERO/OBRA PUBLICA', 
          'TIPO','lat', 'lon', 'fuente', 'query', 'Orgen_data']].to_csv("df_final.csv")